## Day 1: 에이전트 vs. 챗봇, 그리고 ReAct 루프

챗봇은 하나의 메시지를 하나의 답변으로 매핑한다. **에이전트**는 목표에
도달하거나 정지 조건에 걸릴 때까지 관찰 -> 추론 -> 행동 사이클을
반복한다 -- 이 루프가 모든 차이를 만든다(전체 상태 기계 다이어그램과,
observation을 환각이 아닌 실제로 유지시키는 `stop=["Observation:"]`
시퀀스의 메커니즘은 `daily/Day1_...md`를 참고).

이 절에서는 (일일 노트의 식료품 계산 예제와는 다른) 환율 변환 시나리오를
사용해서 같은 세 가지를 처음부터 끝까지 보여준다: 평범한 챗봇이 틀리게
추측하는 것, 실제 도구 호출을 통해 ReAct 루프가 올바르게 답하는 것,
few-shot 프라이밍, 그리고 -- 결정적으로 -- 모델이 `Final Answer`로
수렴하지 못할 때 `max_steps` 가드가 실제로 발동하는 것.

In [ ]:
FEW_SHOT_PRIMER = '''
Question: How many euros is 100 US dollars?
Thought: I need the current USD->EUR rate before I can answer.
Action: convert_currency(100, "USD", "EUR")
Observation: 92.10
Thought: I now have enough to answer.
Final Answer: About 92.10 EUR.
'''

def convert_currency(amount, src, dst):
    # 실제 환율 조회 도구를 대신하는 함수.
    fake_rates = {("USD", "EUR"): 0.921, ("USD", "JPY"): 149.2}
    rate = fake_rates.get((src, dst), 1.0)
    return round(amount * rate, 2)   # -> float

TOOLS = {"convert_currency": convert_currency}

def parse_action(line):
    # 'Action: convert_currency(100, "USD", "EUR")' -> ("convert_currency", (100.0, "USD", "EUR"))
    name = line.split("Action:")[1].split("(")[0].strip()
    raw_args = line.split("(", 1)[1].rsplit(")", 1)[0]
    parts = [p.strip().strip('"') for p in raw_args.split(",")]
    amount, src, dst = float(parts[0]), parts[1], parts[2]
    return name, (amount, src, dst)

def run_currency_agent(fake_model_generate, question, max_steps=5):
    transcript = FEW_SHOT_PRIMER + f"\nQuestion: {question}\n"
    for step in range(max_steps):  # 하드 세이프티 캡: 모델이 스스로 멈추리라고 믿지 않는다
        chunk = fake_model_generate(transcript)
        transcript += chunk
        if "Final Answer:" in chunk:
            return chunk.split("Final Answer:")[-1].strip()
        action_line = next(l for l in chunk.splitlines() if l.startswith("Action:"))
        name, (amount, src, dst) = parse_action(action_line)
        result = TOOLS[name](amount, src, dst)               # 실제 도구 호출, 실제 숫자가 돌아옴
        transcript += f"Observation: {result}\n"             # 하네스가 이 줄을 쓴다 -- 모델이 아니라
    return "Stopped: exceeded max_steps without a Final Answer."

In [ ]:
# 대조를 보여주기 위한 두 가지 모의 모델 동작: 도구 접근이 전혀 없는
# 평범한 챗봇 vs. 실제 convert_currency 도구가 준비된 ReAct 루프에
# 연결된 같은 "모델".

def plain_chatbot_mock(prompt):
    # 도구 접근이 없는 챗봇은 그냥 유창하게 추측한다 -- 그리고 틀릴 수 있다.
    return "Final Answer: Roughly 100 EUR (same as the dollar amount)."

def react_mock_generate(transcript):
    # 실제 로컬 모델 호출을 대신하는 결정적 함수, 예:
    #   inputs = tokenizer(transcript, return_tensors="pt")
    #   out = model.generate(**inputs, max_new_tokens=64, stop_strings=["Observation:"])
    # stop_strings 인자가 모델이 스스로 날조한 Observation 줄을 쓰지
    # 못하게 막는 장치다 -- 일일 노트 참고.
    if "Observation:" not in transcript:
        return 'Thought: I need the rate.\nAction: convert_currency(100, "USD", "EUR")\n'
    return "Thought: Done.\nFinal Answer: 92.10 EUR.\n"

print(plain_chatbot_mock("How many euros is 100 US dollars?"))
print(run_currency_agent(react_mock_generate, "How many euros is 100 US dollars?"))

In [ ]:
# max_steps 가드를 실제로 테스트한다(설명만 하지 않는다): Final Answer를
# 절대 내지 않는 모의 모델 -- 같은 질문을 영원히 다시 묻는다. 루프가
# 계속 도는 대신 실제로 멈추는지 확인한다.

def looping_mock_generate(transcript):
    return 'Thought: let me check the rate again.\nAction: convert_currency(100, "USD", "EUR")\n'

print(run_currency_agent(looping_mock_generate, "How many euros is 100 US dollars?", max_steps=3))
# -> "Stopped: exceeded max_steps without a Final Answer." (검증된 출력)

## Day 2: 에이전트 프레임워크 생태계와 작은 프레임워크

폭넓은 에이전트 도구 생태계는 대략 자율 런타임, 샌드박싱/보안 래퍼,
여러 엔진 위에 올라가는 프레임워크에 독립적인 오케스트레이션 계층으로
나뉜다(비교 다이어그램은 `daily/Day2_...md` 참고). 일일 노트의
`MiniAgentFramework` 예제를 재사용하는 대신, 이 절에서는 문서 번역
시나리오를 중심으로 `ToolRegistry` -- 도구 관리, 불안정한 도구를 위한
재시도 예산, 감사 로그 -- 와 별도의 LCEL 스타일 파이프 체인을 만든다.

In [ ]:
import time

class ToolRegistry:
    """더 큰 에이전트 프레임워크가 관리하는 것을 최소화한 버전: 호출
    가능한 도구들의 테이블(허용/거부 플래그와, 불안정한 도구를 위한
    선택적 재시도 예산 포함)과 계속 쌓이는 감사 로그."""

    def __init__(self):
        self._tools = {}
        self.audit_log = []

    def register_tool(self, name, fn, allowed=True, max_retries=0):
        self._tools[name] = {"fn": fn, "allowed": allowed, "max_retries": max_retries}

    def run_tool(self, name, **kwargs):
        record = {"tool": name, "kwargs": kwargs, "ts": round(time.time(), 3)}
        entry = self._tools.get(name)
        if entry is None or not entry["allowed"]:
            record["status"] = "denied"
            self.audit_log.append(record)
            raise PermissionError(f"tool '{name}' is not available")

        attempts, last_err = 0, None
        while attempts <= entry["max_retries"]:
            attempts += 1
            try:
                result = entry["fn"](**kwargs)          # -> 도구가 반환하는 값
                record["status"] = "ok"
                record["attempts"] = attempts
                record["result"] = result
                self.audit_log.append(record)
                return result
            except Exception as e:                       # 불안정한 도구 호출 실패; 예산이 남으면 재시도
                last_err = e
        record["status"] = "failed_after_retries"
        record["attempts"] = attempts
        record["error"] = str(last_err)
        self.audit_log.append(record)
        raise RuntimeError(f"tool '{name}' failed after {attempts} attempts: {last_err}")

def detect_language(text):
    return "fr" if text.lower().startswith("bonjour") else "en"

# 첫 호출에서는 타임아웃되고 그다음 성공하는 도구 -- 설명이 아니라
# 실제로 재시도 경로를 실행해서 확인한다.
_flaky_state = {"n": 0}
def flaky_translate_api(text):
    _flaky_state["n"] += 1
    if _flaky_state["n"] < 2:
        raise ConnectionError(f"simulated timeout on attempt {_flaky_state['n']}")
    return f"[ES] {text}"

registry = ToolRegistry()
registry.register_tool("detect_language", detect_language)
registry.register_tool("translate", flaky_translate_api, max_retries=2)

print(registry.run_tool("detect_language", text="Bonjour tout le monde"))  # -> "fr"
print(registry.run_tool("translate", text="Hello everyone"))               # -> "[ES] Hello everyone", 재시도 1회 후
print(registry.audit_log)

In [ ]:
# 번역 작업을 위한 개략적인 LCEL 스타일 파이프 체인(prompt | model |
# parser). 이는 메커니즘(함수 합성 위의 평범한 연산자 오버로딩)을
# 보여주기 위한 예시일 뿐, 실제 LangChain 설치가 아니다 -- 실제 연동이라면
# LangChain의 `LLM` 클래스 같은 실제 베이스 클래스를 상속하고 `_call`을
# 구현해야 할 것이다.

class RunnableStep:
    def __or__(self, other):
        return PipedStep(self, other)

class PipedStep(RunnableStep):
    def __init__(self, first, second):
        self.first, self.second = first, second

    def invoke(self, x):
        return self.second.invoke(self.first.invoke(x))

class PromptStep(RunnableStep):
    def __init__(self, template):
        self.template = template

    def invoke(self, variables):
        return self.template.format(**variables)   # -> str

class LocalLLMStep(RunnableStep):
    def __init__(self, generate_fn):
        self.generate_fn = generate_fn

    def invoke(self, prompt_text):
        return self.generate_fn(prompt_text)         # -> str

class StripParserStep(RunnableStep):
    def invoke(self, raw_text):
        return raw_text.strip()                       # -> str, 앞뒤 공백 제거

translate_prompt = PromptStep("Translate to Spanish: {text}")
mock_llm = LocalLLMStep(lambda p: "  Hola a todos  ")
translate_chain = translate_prompt | mock_llm | StripParserStep()
print(translate_chain.invoke({"text": "Hello everyone"}))  # -> "Hola a todos" (공백 제거됨)

## Day 3: 로컬 LLM과 양자화

클라우드 모델은 규모를 얻는 대신 프라이버시와 비용 통제를 내주고, 로컬
모델은 프라이버시, 오프라인 사용, 토큰당 요금 없음을 얻는 대신 규모를
내준다(클라우드-vs-로컬 표 전체와 정밀도 비트 구조 설명은
`daily/Day3_...md` 참고). 이 절에서는 고객지원 티켓 분류 시나리오를 통해
순서대로 보여준다: 서브프로세스로 격리한 측정(이 샌드박스에는
`transformers`/`torch`가 없으므로 numpy로 실제 검증한 메커니즘), 임베딩
테이블에 대한 실제 numpy 메모리 사용량 측정, 이 샌드박스에서 실제로
`except` 분기를 발동시키는 CPU 전용 양자화 폴백, 그리고 모델의 원시
텍스트에서 파싱하는 도구 호출 관례.

In [ ]:
import subprocess
import sys

def measure_alloc_in_subprocess(dtype_name, n_elements):
    """이 dtype의 할당자/캐시 상태가 다음 측정으로 새어 들어가지 않도록
    할당 하나를 자신만의 새 서브프로세스에서 실행한다 -- 실제로는
    AutoModelForCausalLM.from_pretrained(..., torch_dtype=...) 주위에
    쓰는 것과 같은 격리 트릭이다."""
    script = (
        "import time, numpy as np\n"
        "t0 = time.perf_counter()\n"
        f"arr = np.zeros({n_elements}, dtype=np.{dtype_name})\n"
        "print(f'{(time.perf_counter()-t0)*1000:.3f}ms nbytes={arr.nbytes}')\n"
    )
    completed = subprocess.run([sys.executable, "-c", script], capture_output=True, text=True, timeout=30)
    return completed.stdout.strip() if completed.returncode == 0 else f"ERROR: {completed.stderr.strip()}"

for dtype in ["float32", "float16", "int8"]:
    print(dtype, "->", measure_alloc_in_subprocess(dtype, 20_000_000))
# nbytes가 실제 dtype 크기를 확인해준다: 20,000,000개 요소 * 각 4/2/1바이트.

# 이 패턴의 프로덕션 버전은 실제 모델을 대상으로 한다(올바른 API 형태,
# 이 샌드박스에서는 실행하지 않음 -- transformers/torch가 설치되어 있지 않다):
#
# def measure_model_load(model_id, dtype_name):
#     script = (
#         "import time, torch\n"
#         "from transformers import AutoModelForCausalLM\n"
#         "t0 = time.time()\n"
#         f"m = AutoModelForCausalLM.from_pretrained('{model_id}', torch_dtype=torch.{dtype_name})\n"
#         "print(f'{time.time()-t0:.2f}s')\n"
#     )
#     return subprocess.run([sys.executable, "-c", script], capture_output=True, text=True).stdout.strip()

In [ ]:
import numpy as np

# 고객지원 티켓 임베딩 테이블의 실제 메모리 사용량: 50,000단어 어휘 x
# 768차원 임베딩을 실제 numpy .nbytes로 측정한다.
vocab_size, embed_dim = 50_000, 768
n_params = vocab_size * embed_dim
rng = np.random.default_rng(0)

w_fp32 = rng.standard_normal((vocab_size, embed_dim)).astype(np.float32)
w_fp16 = w_fp32.astype(np.float16)                                   # 실제 다운캐스트, 실제 반올림
w_int8 = np.clip(np.round(w_fp32 * 20), -127, 127).astype(np.int8)   # 간단한 affine 양자화 예시

print(f"embedding table: {vocab_size} x {embed_dim} = {n_params:,} params")
for name, arr in [("fp32", w_fp32), ("fp16", w_fp16), ("int8", w_int8)]:
    print(f"  {name}: {arr.nbytes:,} bytes ({arr.nbytes/1024**2:.2f} MiB)")
print(f"  fp32 -> int8 shrink factor: {w_fp32.nbytes / w_int8.nbytes:.2f}x")  # -> 4.00x, 정확히 예상대로

In [ ]:
# CPU 전용 폴백 사례 연구: bitsandbytes + CUDA 백엔드가 필요한 8비트
# 양자화 단계가, 둘 다 없을 때(이 샌드박스처럼) 스크립트 전체를 죽이지
# 않고 우아하게 실패해야 하는 경우 -- 아래 except 분기가 실제로 실행된다.

def build_quantized_layer(in_features, out_features):
    try:
        import bitsandbytes as bnb
        return bnb.nn.Linear8bitLt(in_features, out_features, has_fp16_weights=False)
    except Exception as e:
        print(f"quantized layer unavailable ({type(e).__name__}: {e}); using full precision instead")
        return f"fallback_linear({in_features}x{out_features})"  # torch.nn.Linear를 대신하는 값

layer = build_quantized_layer(512, 512)
print("layer:", layer)

In [ ]:
import re

# 도구 호출 관례: 모델이 아래와 같은 줄을 내도록 지시받는다
#   TOOL_CALL: lookup_order_status(order_id=4471)
# 그러면 Python이 정규식으로 이를 추출하고 디스패치한다. 작은 로컬
# 모델은 형식이 올바르고 이스케이프도 정확한 JSON을 내는 것보다 이런
# 고정된 템플릿을 재현하는 데 훨씬 더 안정적이다.
TOOL_CALL_PATTERN = re.compile(r"TOOL_CALL:\s*(\w+)\((.*)\)")

def lookup_order_status(order_id):
    return {"order_id": order_id, "status": "shipped"}

SUPPORT_TOOLS = {"lookup_order_status": lookup_order_status}

def dispatch_from_model_text(model_text):
    match = TOOL_CALL_PATTERN.search(model_text)
    if not match:
        return None                                    # 도구 호출 없음 -- 이것도 정상적인 경우
    tool_name, raw_args = match.group(1), match.group(2)
    order_id = int(raw_args.split("=")[1])
    return SUPPORT_TOOLS[tool_name](order_id=order_id)

print(dispatch_from_model_text("TOOL_CALL: lookup_order_status(order_id=4471)"))  # -> {'order_id': 4471, 'status': 'shipped'}
print(dispatch_from_model_text("Let me think about this..."))                     # -> None, 오류가 아니다

## Day 4: 에이전트 안전장치

도구 허용 목록(default-deny), 스텝 제한, 사람 승인 게이트, 비용 상한이라는
네 가지 안전장치를 하나의 가드 러너로 결합하고, 가장 저렴한 검사부터
확인한다(전체 결정 흐름 다이어그램은 `daily/Day4_...md` 참고). 이 절은
소셜 미디어/환불 처리 에이전트를 사용하며, 결정적으로 **올바른 순서**의
게이트와 **버그 있는 순서**의 게이트를 동일한 시도 시퀀스에 대해
실행해서, 검사 순서가 최종 예산뿐 아니라 거부 사유로 무엇이 기록되는지도
바꾼다는 사실을 단순히 주장이 아니라 실제로 보여준다.

In [ ]:
import time

class SafeAgentGate:
    """네 가지 안전장치를 가장 저렴하고 빠른 것부터 검사하는 순서로
    결합한다. 그리고 -- 결정적으로 -- step_count/spent_usd는 모든 검사가
    이미 통과한 이후, 최종 `allowed` 분기에서만 커밋한다."""

    def __init__(self, allowed_tools, max_steps, budget_usd, approve_fn):
        self.allowed_tools = set(allowed_tools)
        self.max_steps = max_steps
        self.budget_usd = budget_usd
        self.spent_usd = 0.0
        self.steps_taken = 0
        self.approve_fn = approve_fn
        self.audit_rows = []

    def attempt(self, tool_name, args, est_cost, needs_approval=False):
        row = {"timestamp": round(time.time(), 3), "tool": tool_name, "args": str(args),
               "cost": est_cost, "decision": None}
        if self.steps_taken >= self.max_steps:
            row["decision"] = "denied_step_limit"
        elif self.spent_usd + est_cost > self.budget_usd:
            row["decision"] = "denied_cost_cap"
        elif tool_name not in self.allowed_tools:
            row["decision"] = "denied_not_allowlisted"
        elif needs_approval and not self.approve_fn(tool_name, args):
            row["decision"] = "denied_no_approval"
        else:
            row["decision"] = "allowed"
            self.steps_taken += 1
            self.spent_usd += est_cost      # 모든 검사가 통과했을 때만 커밋
        self.audit_rows.append(row)
        return row["decision"] == "allowed"


class BuggySafeAgentGate(SafeAgentGate):
    """같은 네 가지 검사지만 순서가 잘못됐다: 허용 목록 검사가 실행되기
    전에 비용이 커밋되므로, 허용되지 않은 도구의 추정 비용이 실제로
    실행되지도 않았는데 예산에 청구된다."""

    def attempt(self, tool_name, args, est_cost, needs_approval=False):
        row = {"timestamp": round(time.time(), 3), "tool": tool_name, "args": str(args),
               "cost": est_cost, "decision": None}
        if self.spent_usd + est_cost > self.budget_usd:
            row["decision"] = "denied_cost_cap"
            self.audit_rows.append(row)
            return False
        self.spent_usd += est_cost   # 버그: 아래의 허용 목록 검사보다 먼저 커밋됨
        if tool_name not in self.allowed_tools:
            row["decision"] = "denied_not_allowlisted"
            self.audit_rows.append(row)
            return False
        row["decision"] = "allowed"
        self.steps_taken += 1
        self.audit_rows.append(row)
        return True


def post_to_social_media(caption):
    return f"posted: {caption}"

def refund_customer(order_id, amount):
    return f"refunded {amount} for order {order_id}"

def always_deny(tool_name, args):
    return False  # 실제 사람 승인 프롬프트를 대신하는 함수

In [ ]:
# 막힌 에이전트가 같은 거부된 호출을 두 번 반복 시도한 뒤, 정당한
# 호출을 한 번 하는 상황을 -- 동일한 입력으로 두 게이트 모두에 대해 실행한다.

def run_scenario(gate_cls):
    gate = gate_cls(allowed_tools={"post_to_social_media"}, max_steps=10, budget_usd=1.00, approve_fn=always_deny)
    gate.attempt("refund_customer", {"order_id": 91, "amount": 0.40}, est_cost=0.40, needs_approval=True)
    gate.attempt("refund_customer", {"order_id": 91, "amount": 0.40}, est_cost=0.40, needs_approval=True)
    gate.attempt("post_to_social_media", {"caption": "New product!"}, est_cost=0.02)
    return gate

correct_gate = run_scenario(SafeAgentGate)
buggy_gate = run_scenario(BuggySafeAgentGate)

print("=== correct order ===")
for r in correct_gate.audit_rows:
    print(r)
print(f"spent_usd = {correct_gate.spent_usd:.2f}\n")

print("=== buggy order (cost committed before allowlist check) ===")
for r in buggy_gate.audit_rows:
    print(r)
print(f"spent_usd = {buggy_gate.spent_usd:.2f}")

assert round(correct_gate.spent_usd, 2) == 0.02   # 허용된 게시물만 실제로 지출을 발생시켰다
assert round(buggy_gate.spent_usd, 2) == 0.82      # 한 번도 허용된 적 없는 환불 시도 두 번도 각 0.40씩 "지출"됨
print("\nconfirmed: buggy order over-counts spend for calls that were never actually allowed")

In [ ]:
import pandas as pd

audit_df = pd.DataFrame(correct_gate.audit_rows)

# 비정상적으로 많이 시도된 (tool, args) 조합을 표시한다 -- 막히거나
# 오작동하는 에이전트를 잡아내는 단순한 휴리스틱이며, ML 모델이 필요
# 없다. 여기서는 반복되는, 동일한, 거부된 refund_customer 시도가
# 정확히 이 검사가 잡아내야 할 종류의 것이다.
repeat_counts = (
    audit_df.groupby(["tool", "args"])
    .size()
    .reset_index(name="attempts")
)
flagged = repeat_counts[repeat_counts["attempts"] >= 2]
print(audit_df)
print("\nflagged:\n", flagged)

# 모델 라우팅 패턴(정성적으로만, 지어낸 수치 없이): 일상적인 단계는
# 저렴한 로컬 모델로 보내고, 로컬 에이전트가 막히거나 스텝을 다 썼을
# 때만 마지막으로 한 번 더 큰 클라우드 모델로 넘어간다 -- 대부분의
# 요청은 저렴한 경로에 머물고, 어려운 케이스도 더 강력한 모델을 한 번
# 시도해볼 기회를 얻는다. 이 상향 호출도 위와 같은 안전장치를 그대로
# 통과해야 한다 -- "비싼 폴백"이라는 사실이 비용 상한을 건너뛸 이유는
# 아니다.
def route_request(local_agent_run, cloud_agent_run, question):
    local_result = local_agent_run(question)
    if local_result is None:
        return cloud_agent_run(question)  # 막혔을 때만 상향 이동
    return local_result